# Load CSV 

Firstly we need to load our data from yahoo Finance database, 
We will choose several parameters (Start Date, End Date, TICKER) In order to lead our study.


In [4]:
import datetime as dt
import pandas as pd
import yfinance as yf
from pathlib import Path
import os

TICKER = "AAPL"
START = "2024-01-01"
END = "2026-01-01"

try:
    base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    base_dir = os.getcwd()

OUT_DIR = os.path.join(base_dir, "data")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_FILE = os.path.join(OUT_DIR, f"{TICKER}_{START[:4]}.csv")

Then we need to decide Which information we will use for our study.
We will follow closely : 
-        The date 
-        The opening value of the day
-        The highest value
-        The lowest value
-        The ending value of the day
-        The djusted closing price (closing value adjusted after dividends, splits...)
-        The daily volume of transaction

Then, we need to convert our Csv File into a pandas dataframe.
We define load_scv()


In [5]:
def loadcsv():
    if os.path.exists(OUT_FILE):
        print(f"Data already exists in {OUT_FILE}. Loading locally...")
        return pd.read_csv(OUT_FILE)

    import requests
    session = requests.Session()
    session.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
    
    print(f"Downloading {TICKER} from {START} to {END}")
    df = yf.download(TICKER, start = START, end = END, auto_adjust = False, session=session)

    if df.empty :
        raise SystemExit("No data received")

    df = df.reset_index()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    print("Columns after flatten:", list(df.columns))  # debug

    rename_map = {
        "Date": "Date",
        "Open": "Open",
        "High": "High",
        "Low": "Low",
        "Close": "Close",
        "Adj Close": "AdjClose",
        "Volume": "Volume",
    }
    df = df.rename(columns=rename_map)

    if "AdjClose" not in df.columns and "Close" in df.columns:
        df["AdjClose"] = df["Close"]

    wanted = ["Date", "Open", "High", "Low", "Close", "AdjClose", "Volume"]
    present = [c for c in wanted if c in df.columns]
    df = df[present]

    subset_clean = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in df.columns]
    if subset_clean:
        df = df.dropna(subset=subset_clean)
        if "Volume" in df.columns:
            df = df[df["Volume"] > 0]

    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors = "coerce").dt.date.astype(str)
        df = df.sort_values("Date")

    df.to_csv(OUT_FILE, index=False)
    print(f"File saved -> {os.path.abspath(OUT_FILE)}")
    print("\nPreview:")
    print(df.head().to_string(index=False))
    print("\nInfo:")
    print(f"Rows: {len(df)} | Period: {df['Date'].min()} -> {df['Date'].max()}")
    print("Columns:", ", ".join(df.columns))

    return (df)

In [7]:
df = loadcsv()
print(df.head())

[*********************100%***********************]  1 of 1 completed

1 Failed download:
['AAPL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


SystemExit: No data received

/Users/robincrifo/.local/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Conclusion of Notebook 1

We succeed to load our data from Yahoo Finance, now we will start to implement our models to predict the value of the TICKER.